## Example 1: SFNClient Type Annotations for Step Functions Management

In [ ]:
from types_aiobotocore_stepfunctions import SFNClient
from types_aiobotocore_stepfunctions.type_defs import ListStateMachinesOutputTypeDef, StateMachineListItemTypeDef, DescribeStateMachineOutputTypeDef
from aiobotocore.session import get_session
from botocore.exceptions import NoCredentialsError

async def demonstrate_sfn_client_types():
    session = get_session()
    async with session.create_client("stepfunctions", region_name="us-east-1") as client:
        client: SFNClient  # Type annotation enables IDE autocomplete
        try:
            state_machines_response: ListStateMachinesOutputTypeDef = await client.list_state_machines(maxResults=10)
            print(f"Step Functions State Machines: {len(state_machines_response.get('stateMachines', []))}")
            for sm in state_machines_response.get("stateMachines", [])[:3]:
                sm_typed: StateMachineListItemTypeDef = sm
                print(f"  State Machine: {sm_typed.get('name')} (Type: {sm_typed.get('type')})")
            if state_machines_response.get('stateMachines'):
                sm_arn = state_machines_response['stateMachines'][0]['stateMachineArn']
                describe_response: DescribeStateMachineOutputTypeDef = await client.describe_state_machine(stateMachineArn=sm_arn)
                print(f"\nState Machine Details: {describe_response.get('name')}")
        except NoCredentialsError:
            print("No AWS credentials (expected)")
        except Exception as e:
            print(f"No state machines/access denied (expected): {type(e).__name__}")
        print("SFNClient provides 40+ typed methods for Step Functions management")

await demonstrate_sfn_client_types()

## Example 2: Using Paginator Types for Step Functions Resources

In [ ]:
from types_aiobotocore_stepfunctions import SFNClient
from types_aiobotocore_stepfunctions.paginator import ListStateMachinesPaginator, ListExecutionsPaginator, GetExecutionHistoryPaginator
from aiobotocore.session import get_session
from botocore.exceptions import NoCredentialsError

async def demonstrate_sfn_paginator_types():
    session = get_session()
    async with session.create_client("stepfunctions", region_name="us-east-1") as client:
        client: SFNClient
        state_machines_paginator: ListStateMachinesPaginator = client.get_paginator("list_state_machines")
        executions_paginator: ListExecutionsPaginator = client.get_paginator("list_executions")
        history_paginator: GetExecutionHistoryPaginator = client.get_paginator("get_execution_history")
        print("Step Functions Paginator types:")
        print(f"  ListStateMachinesPaginator: {type(state_machines_paginator).__name__}")
        print(f"  ListExecutionsPaginator: {type(executions_paginator).__name__}")
        print(f"  GetExecutionHistoryPaginator: {type(history_paginator).__name__}")
        try:
            sm_count = 0
            async for page in state_machines_paginator.paginate():
                sm_count += len(page.get("stateMachines", []))
            print(f"Found {sm_count} state machines using paginator")
        except (NoCredentialsError, Exception):
            print("No credentials/state machines (expected)")
        print("Package provides 5+ Step Functions paginator types")

await demonstrate_sfn_paginator_types()

## Example 3: Using TypeDefs for State Machine Creation

In [ ]:
from types_aiobotocore_stepfunctions.type_defs import CreateStateMachineInputTypeDef, TagTypeDef, LoggingConfigurationTypeDef, CreateStateMachineOutputTypeDef
import json

def demonstrate_state_machine_typedefs():
    tags: list[TagTypeDef] = [{"key": "Environment", "value": "Production"}, {"key": "Application", "value": "OrderProcessing"}]
    logging_config: LoggingConfigurationTypeDef = {"level": "ALL", "includeExecutionData": True, "destinations": [{"cloudWatchLogsLogGroup": {"logGroupArn": "arn:aws:logs:us-east-1:123456789012:log-group:/aws/stepfunctions/orders"}}]}
    definition = {"Comment": "Order processing workflow", "StartAt": "ProcessOrder", "States": {"ProcessOrder": {"Type": "Task", "Resource": "arn:aws:lambda:us-east-1:123456789012:function:ProcessOrder", "End": True}}}
    create_state_machine: CreateStateMachineInputTypeDef = {
        "name": "OrderProcessingStateMachine",
        "definition": json.dumps(definition),
        "roleArn": "arn:aws:iam::123456789012:role/StepFunctionsExecutionRole",
        "type": "STANDARD",
        "loggingConfiguration": logging_config,
        "tags": tags
    }
    print("Type-safe State Machine Creation:")
    print(f"  Name: {create_state_machine['name']}")
    print(f"  Type: {create_state_machine['type']}")
    print(f"  Logging Level: {logging_config['level']}")
    print(f"  Tags: {len(tags)} tags")
    response: CreateStateMachineOutputTypeDef = {"stateMachineArn": "arn:aws:states:us-east-1:123456789012:stateMachine:OrderProcessingStateMachine", "creationDate": "2024-01-15T10:00:00Z"}
    print(f"\nState Machine Created: {response['stateMachineArn'][-50:]}")
    print("TypeDefs ensure type safety for Step Functions state machine creation")

demonstrate_state_machine_typedefs()

## Example 4: Using TypeDefs for Execution Management

In [ ]:
from types_aiobotocore_stepfunctions.type_defs import StartExecutionInputTypeDef, StartExecutionOutputTypeDef, DescribeExecutionOutputTypeDef
import json

def demonstrate_execution_typedefs():
    input_data = {"orderId": "12345", "customerId": "customer-abc", "items": [{"sku": "ITEM-001", "quantity": 2}], "totalAmount": 59.99}
    start_execution: StartExecutionInputTypeDef = {
        "stateMachineArn": "arn:aws:states:us-east-1:123456789012:stateMachine:OrderProcessingStateMachine",
        "name": "order-12345-execution",
        "input": json.dumps(input_data)
    }
    print("Type-safe Execution Start Request:")
    print(f"  State Machine: {start_execution['stateMachineArn'][-40:]}")
    print(f"  Execution Name: {start_execution['name']}")
    print(f"  Input Data: Order {input_data['orderId']}, Items: {len(input_data['items'])}")
    start_response: StartExecutionOutputTypeDef = {
        "executionArn": "arn:aws:states:us-east-1:123456789012:execution:OrderProcessingStateMachine:order-12345-execution",
        "startDate": "2024-01-15T10:00:00Z"
    }
    describe_response: DescribeExecutionOutputTypeDef = {
        "executionArn": start_response['executionArn'],
        "stateMachineArn": start_execution['stateMachineArn'],
        "status": "RUNNING",
        "startDate": start_response['startDate']
    }
    print(f"\nExecution Started: {start_response['executionArn'][-50:]}")
    print(f"Status: {describe_response['status']}")
    print("TypeDefs ensure type safety for Step Functions execution management")

demonstrate_execution_typedefs()

## Example 5: Using TypeDefs for Activity Management

In [ ]:
from types_aiobotocore_stepfunctions.type_defs import CreateActivityInputTypeDef, CreateActivityOutputTypeDef, GetActivityTaskOutputTypeDef, SendTaskSuccessInputTypeDef
import json

def demonstrate_activity_typedefs():
    tags: list = [{"key": "Environment", "value": "Production"}, {"key": "WorkerType", "value": "ImageProcessor"}]
    create_activity: CreateActivityInputTypeDef = {
        "name": "ProcessImageActivity",
        "tags": tags
    }
    print("Type-safe Activity Creation:")
    print(f"  Activity Name: {create_activity['name']}")
    print(f"  Tags: {len(tags)} tags")
    activity_response: CreateActivityOutputTypeDef = {
        "activityArn": "arn:aws:states:us-east-1:123456789012:activity:ProcessImageActivity",
        "creationDate": "2024-01-15T10:00:00Z"
    }
    task_response: GetActivityTaskOutputTypeDef = {
        "taskToken": "AAAAKgAAAAIAAAAAAAAAAboB...",
        "input": json.dumps({"imageUrl": "s3://bucket/image.jpg", "filters": ["resize", "watermark"]})
    }
    success_input: SendTaskSuccessInputTypeDef = {
        "taskToken": task_response['taskToken'],
        "output": json.dumps({"resultUrl": "s3://bucket/processed-image.jpg", "status": "completed"})
    }
    print(f"\nActivity Created: {activity_response['activityArn'][-40:]}")
    print(f"Task Token: {task_response['taskToken'][:20]}...")
    print(f"Task Input: {json.loads(task_response['input'])['imageUrl']}")
    print("TypeDefs ensure type safety for Step Functions activity management")

demonstrate_activity_typedefs()